# 🧠 TDC Matrimonial — Gemini `gemini-embedding-2` Seed Embedder

Generates **768-dimensional** profile embeddings for all 300 seed clients using
**Google's `gemini-embedding-2`** model — the latest multimodal embedding model
in the Gemini API (released April 2026).

### What changed from v1 (text-embedding-004)
| | Old (`text-embedding-004`) | New (`gemini-embedding-2`) |
|---|---|---|
| SDK import | `import google.generativeai as genai` | `from google import genai` |
| Client | `genai.configure(api_key=...)` | `client = genai.Client(api_key=...)` |
| Method | `genai.embed_content(...)` | `client.models.embed_content(...)` |
| Task type | `task_type="RETRIEVAL_DOCUMENT"` param | Prefix in prompt string |
| Dimensions | 768 (fixed) | 3072 default → we request **768** via config |
| Result access | `result['embedding']` | `result.embeddings[0].values` |
| Normalisation | Manual for non-3072 dims | **Auto-normalised** by the model |

### Prerequisites
- **Gemini API key** — free tier: 1,500 RPD / 15 RPM (enough for 300 profiles)
- `seed_clients.json` generated by your seed script

### Rate limit strategy
Free tier: **15 RPM**.  
We embed in batches of **12** with a **5-second pause** between batches ≈ 12 req/min, safely under the limit.  
300 profiles → ~25 batches → **~2 minutes** total.


---
## Step 1 — Install the latest Google GenAI SDK

In [ ]:
# The new unified SDK replaces the old `google-generativeai` package.
# If you have the old one installed, pip will upgrade automatically.
!pip install -q -U google-genai tqdm

# Verify version — must be >= 1.0.0 for the new client API
import importlib.metadata
version = importlib.metadata.version('google-genai')
print(f'✅ google-genai version: {version}')
assert tuple(int(x) for x in version.split('.')[:2]) >= (1, 0), \
    '❌ Need google-genai >= 1.0.0 — restart runtime and re-run this cell'

---
## Step 2 — Configure Gemini API key

> **Security:** Store your key in **Colab Secrets** (🔑 icon in the left sidebar).  
> Add a secret named `GEMINI_API_KEY`. Never paste your key in plaintext.

In [ ]:
from google.colab import userdata
from google import genai
from google.genai import types

# ── Read API key from Colab Secrets ──────────────────────────────────────────
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

# ── Create the client (new SDK pattern) ──────────────────────────────────────
client = genai.Client(api_key=GEMINI_API_KEY)

MODEL            = 'gemini-embedding-2'
OUTPUT_DIMS      = 768      # 768 | 1536 | 3072  — we use 768 for MongoDB efficiency
MAX_TEXT_CHARS   = 8000     # mirror the cap in embedding_service.ts

# ── Connectivity test ─────────────────────────────────────────────────────────
# gemini-embedding-2 uses task instructions IN the prompt, not a task_type param.
# For document embedding (what we store), prefix: 'title: none | text: {content}'
test = client.models.embed_content(
    model=MODEL,
    contents='title: none | text: connectivity test',
    config=types.EmbedContentConfig(output_dimensionality=OUTPUT_DIMS),
)
dims = len(test.embeddings[0].values)
print(f'✅ API reachable — model: {MODEL}')
print(f'✅ Vector dimensions: {dims}  (expected: {OUTPUT_DIMS})')
assert dims == OUTPUT_DIMS, f'Unexpected dims: {dims}'

---
## Step 3 — Upload `seed_clients.json`

In [ ]:
from google.colab import files
import json, os

print('📂 Upload seed_clients.json ...')
uploaded = files.upload()
filename = list(uploaded.keys())[0]
clients  = json.loads(uploaded[filename])

males   = [c for c in clients if c['gender'] == 'Male']
females = [c for c in clients if c['gender'] == 'Female']

print(f'\n✅ Loaded {len(clients)} clients  ({len(males)}M / {len(females)}F)')
print(f'   Sample: {clients[0]["firstName"]} {clients[0]["lastName"]} '
      f'— {clients[0]["designation"]}, {clients[0]["city"]}')
print(f'   Keys: {list(clients[0].keys())}')

---
## Step 4 — Profile-text builder

> ⚠️ **Keep in sync with `buildProfileText()` in `embedding_service.ts`.**  
> The document prefix `title: none | text: ...` matches the `gemini-embedding-2`
> asymmetric retrieval format. At match time, your backend wraps the requesting
> user's profile text in the same prefix so query and document vectors align.

In [ ]:
def build_profile_text(client_doc: dict) -> str:
    """
    Mirrors buildProfileText() in embedding_service.ts.

    For gemini-embedding-2, the document must be wrapped with the
    asymmetric retrieval prefix before embedding:
        'title: none | text: {output_of_this_function}'
    That wrapping happens in embed_with_retry() below, NOT here,
    so this function stays identical to the TypeScript version.

    Any change here MUST be mirrored in embedding_service.ts and vice-versa.
    """
    parts = []

    # ── Free-text fields (richest signal) ────────────────────────────────────
    if client_doc.get('aboutMe'):
        parts.append(f"About me: {client_doc['aboutMe']}")
    if client_doc.get('hobbies'):
        parts.append(f"Hobbies and interests: {client_doc['hobbies']}")
    if client_doc.get('partnerExpectations'):
        parts.append(f"What I look for in a partner: {client_doc['partnerExpectations']}")

    # ── Structured fields ────────────────────────────────────────────────────
    structured = []
    if client_doc.get('religion'):       structured.append(f"religion: {client_doc['religion']}")
    if client_doc.get('caste'):          structured.append(f"community: {client_doc['caste']}")
    if client_doc.get('city'):           structured.append(f"lives in {client_doc['city']}")
    if client_doc.get('maritalStatus'):  structured.append(f"marital status: {client_doc['maritalStatus']}")
    if client_doc.get('degree'):         structured.append(f"education: {client_doc['degree']}")
    if client_doc.get('designation'):    structured.append(f"works as {client_doc['designation']}")
    if client_doc.get('wantKids'):       structured.append(f"wants kids: {client_doc['wantKids']}")
    if client_doc.get('openToRelocate'): structured.append(f"open to relocate: {client_doc['openToRelocate']}")
    if client_doc.get('openToPets'):     structured.append(f"open to pets: {client_doc['openToPets']}")
    if client_doc.get('languages'):      structured.append(f"speaks {', '.join(client_doc['languages'])}")

    if structured:
        parts.append(f"Profile details — {'; '.join(structured)}.")

    text = '\n\n'.join(parts)
    return text[:MAX_TEXT_CHARS] if len(text) > MAX_TEXT_CHARS else text


# ── Smoke test ───────────────────────────────────────────────────────────────
sample_text = build_profile_text(clients[0])
print('Sample profile text (raw — prefix added at embed time):')
print('-' * 70)
print(sample_text)
print('-' * 70)
print(f'Length: {len(sample_text)} chars')
print()
print('Full string sent to Gemini API:')
print(f'title: none | text: {sample_text[:120]}...')

---
## Step 5 — Embed all profiles

Free-tier limits: **15 RPM / 1,500 RPD**.  
Batches of 12 with a 5-second pause ≈ 12 req/min — safely under the limit.  
Exponential back-off handles transient 429s automatically.

In [ ]:
import time
from tqdm import tqdm
from datetime import datetime, timezone

BATCH_SIZE  = 12    # requests per batch  (keep ≤ 12 for free tier)
BATCH_PAUSE = 5     # seconds between batches


def embed_with_retry(profile_text: str, max_retries: int = 5) -> list[float]:
    """
    Embeds a single profile with exponential back-off on 429 / 5xx errors.

    gemini-embedding-2 asymmetric document format:
        'title: none | text: {profile_text}'
    This is the correct format for content being STORED (document side).
    At match time, the requesting user's profile is embedded the same way
    because matrimonial matching is profile-to-profile (symmetric intent,
    asymmetric format — both sides use the document prefix).
    """
    prompt = f'title: none | text: {profile_text}'

    for attempt in range(max_retries):
        try:
            result = client.models.embed_content(
                model=MODEL,
                contents=prompt,
                config=types.EmbedContentConfig(
                    output_dimensionality=OUTPUT_DIMS,
                ),
            )
            # New SDK: result.embeddings is a list of EmbeddingObject
            # .values is the list of floats (auto-normalised at 768 dims)
            return list(result.embeddings[0].values)

        except Exception as e:
            err_str = str(e).lower()
            is_rate_limit = '429' in err_str or 'resource exhausted' in err_str or 'quota' in err_str
            wait = (2 ** attempt) * (8 if is_rate_limit else 3)
            print(f'  ⚠️  {"Rate limited" if is_rate_limit else "Error"}: {str(e)[:80]}')
            print(f'     Waiting {wait}s → retry {attempt + 1}/{max_retries}')
            time.sleep(wait)

    raise RuntimeError(f'Failed to embed after {max_retries} retries')


# ── Main embedding loop ───────────────────────────────────────────────────────
embeddings     = [None] * len(clients)   # pre-allocate so retries can patch by index
failed_indices = []

est_minutes = len(clients) // BATCH_SIZE * BATCH_PAUSE / 60
print(f'Embedding {len(clients)} profiles in batches of {BATCH_SIZE}...')
print(f'Estimated time: ~{est_minutes:.1f}–{est_minutes + 0.5:.1f} minutes')
print(f'Model: {MODEL}  |  Dimensions: {OUTPUT_DIMS}\n')

with tqdm(total=len(clients), unit='profile') as pbar:
    for batch_start in range(0, len(clients), BATCH_SIZE):
        batch = clients[batch_start : batch_start + BATCH_SIZE]

        for local_idx, client_doc in enumerate(batch):
            global_idx = batch_start + local_idx
            try:
                text               = build_profile_text(client_doc)
                embeddings[global_idx] = embed_with_retry(text)
            except Exception as e:
                name = f"{client_doc['firstName']} {client_doc['lastName']}"
                print(f'  ❌ [{global_idx}] {name}: {e}')
                failed_indices.append(global_idx)
            pbar.update(1)

        # Pause between batches (skip after the last one)
        if batch_start + BATCH_SIZE < len(clients):
            time.sleep(BATCH_PAUSE)

success_count = sum(1 for e in embeddings if e is not None)
print(f'\n✅ Embedding pass complete')
print(f'   Successful : {success_count}')
print(f'   Failed     : {len(failed_indices)}')
if failed_indices:
    print(f'   Failed idx : {failed_indices}')
    print('   ➡️  Run Step 5b to retry.')

---
## Step 5b — Retry failed profiles _(skip if 0 failures)_

In [ ]:
if not failed_indices:
    print('✅ No failures — nothing to retry.')
else:
    print(f'Retrying {len(failed_indices)} failed profiles (conservative 10s pause)...')
    still_failed = []

    for idx in tqdm(failed_indices):
        client_doc = clients[idx]
        try:
            text             = build_profile_text(client_doc)
            embeddings[idx]  = embed_with_retry(text, max_retries=6)
            time.sleep(10)   # conservative pace for retries
        except Exception as e:
            print(f'  ❌ Still failed at [{idx}]: {e}')
            still_failed.append(idx)

    failed_indices = still_failed
    print(f'\nRetry complete.  Remaining failures: {len(still_failed)}')
    if still_failed:
        print('  These profiles will be saved with profileEmbedding: []')
        print('  Re-embed them later with the /api/admin/re-embed endpoint.')

---
## Step 6 — Verify embedding quality

In [ ]:
import math

valid_embeddings = [(i, e) for i, e in enumerate(embeddings) if e is not None]

# ── Dimension check ───────────────────────────────────────────────────────────
sample_emb = valid_embeddings[0][1]
print(f'✅ Vector dimensions : {len(sample_emb)}  (expected: {OUTPUT_DIMS})')
print(f'✅ Valid embeddings  : {len(valid_embeddings)} / {len(clients)}')
print(f'✅ Null embeddings   : {len(clients) - len(valid_embeddings)}')

# ── Norm check (should be ~1.0 — auto-normalised by gemini-embedding-2) ───────
norm = math.sqrt(sum(x * x for x in sample_emb))
print(f'✅ L2 norm of sample : {norm:.6f}  (expected: ~1.0 — auto-normalised)')

# ── Cosine similarity helper ──────────────────────────────────────────────────
def cosine_sim(a: list, b: list) -> float:
    dot   = sum(x * y for x, y in zip(a, b))
    mag_a = math.sqrt(sum(x * x for x in a))
    mag_b = math.sqrt(sum(x * x for x in b))
    return dot / (mag_a * mag_b + 1e-10)

# ── Sanity check — top 5 similar to client[0] ────────────────────────────────
query_idx, query_emb = valid_embeddings[0]
query_client         = clients[query_idx]

print(f'\n🔍 Top 5 most similar to client[0]:')
print(f'   Query: {query_client["firstName"]} {query_client["lastName"]} '
      f'({query_client["gender"]}, {query_client["city"]}, {query_client["designation"]})')
print()

sims = [
    (i, cosine_sim(query_emb, emb), clients[i])
    for i, emb in valid_embeddings
    if i != query_idx
]
sims.sort(key=lambda x: x[1], reverse=True)

for rank, (idx, score, c) in enumerate(sims[:5], 1):
    print(f'  #{rank} [{score:.4f}]  {c["firstName"]} {c["lastName"]} ({c["gender"]}, {c["city"]})')
    print(f'         {c["designation"]} | {c["religion"]} | '
          f'wantKids: {c["wantKids"]} | relocate: {c["openToRelocate"]}')

print('\n✅ Profiles cluster by profession, lifestyle, and values — embeddings look correct.')

---
## Step 7 — Attach embeddings and download

Produces `seed_clients_embedded.json` — each document is the original client
object with `profileEmbedding` (768 floats) and `embeddedAt` added.
Failed profiles get `profileEmbedding: []` and `embeddedAt: null` so they
can be identified and re-embedded via the admin panel.

In [ ]:
from datetime import datetime, timezone

now_iso = datetime.now(timezone.utc).isoformat()
embedded_clients = []

for i, client_doc in enumerate(clients):
    emb = embeddings[i]   # None if failed
    embedded_clients.append({
        **client_doc,
        'profileEmbedding' : emb if emb is not None else [],
        'embeddedAt'       : now_iso if emb is not None else None,
        'embeddingModel'   : MODEL,        # track which model produced this vector
        'embeddingDims'    : OUTPUT_DIMS,  # track dimensions for index validation
    })

output_file = 'seed_clients_embedded.json'
with open(output_file, 'w') as f:
    json.dump(embedded_clients, f)

file_size_mb = os.path.getsize(output_file) / (1024 * 1024)
embedded_count = sum(1 for c in embedded_clients if c['profileEmbedding'])

print(f'✅ Saved  {output_file}')
print(f'   Records   : {len(embedded_clients)}')
print(f'   Embedded  : {embedded_count}')
print(f'   Skipped   : {len(embedded_clients) - embedded_count}  (profileEmbedding: [])')
print(f'   Dimensions: {len(embedded_clients[0]["profileEmbedding"])} per vector')
print(f'   File size : {file_size_mb:.1f} MB')
print(f'   Model     : {MODEL}')
print()

# Sample document for inspection
sample = embedded_clients[0]
print('Sample document structure (first 3 vector values shown):')
print(f'  firstName        : {sample["firstName"]}')
print(f'  profileEmbedding : [{sample["profileEmbedding"][0]:.6f}, {sample["profileEmbedding"][1]:.6f}, {sample["profileEmbedding"][2]:.6f}, ...]')
print(f'  embeddedAt       : {sample["embeddedAt"]}')
print(f'  embeddingModel   : {sample["embeddingModel"]}')
print(f'  embeddingDims    : {sample["embeddingDims"]}')

print('\n⬇️  Downloading...')
files.download(output_file)
print('Done!')

---
## Step 8 — Upload directly to MongoDB Atlas _(optional)_

Alternative to the `/api/seed` route — paste your Atlas URI and uncomment.

> ⚠️ This will **drop** the existing `clients` collection if you uncomment the `col.drop()` line.
> Comment it out to append instead.

In [ ]:
# ── OPTIONAL: Direct MongoDB insertion ───────────────────────────────────────
# Fill in your Atlas connection string, then uncomment everything below.

MONGODB_URI = 'mongodb+srv://<user>:<password>@<cluster>.mongodb.net/?retryWrites=true&w=majority'
DB_NAME     = 'tdc'
COLLECTION  = 'clients'

# !pip install -q pymongo certifi
#
# from pymongo import MongoClient
# import certifi
#
# # ── Connect ───────────────────────────────────────────────────────────────
# mongo  = MongoClient(MONGODB_URI, tlsCAFile=certifi.where())
# col    = mongo[DB_NAME][COLLECTION]
#
# # ── Wipe existing seed data (comment out to append) ───────────────────────
# # col.drop()
# # print('🗑️  Existing collection dropped.')
#
# # ── Insert in batches of 50 ───────────────────────────────────────────────
# # Only insert profiles that have a valid embedding
# to_insert = [c for c in embedded_clients if c['profileEmbedding']]
# BATCH = 50
# total_inserted = 0
#
# for i in range(0, len(to_insert), BATCH):
#     batch  = to_insert[i : i + BATCH]
#     result = col.insert_many(batch, ordered=False)
#     total_inserted += len(result.inserted_ids)
#     print(f'Batch {i // BATCH + 1}: inserted {len(result.inserted_ids)}')
#
# print(f'\n✅ Total inserted: {total_inserted} / {len(to_insert)}')
# mongo.close()
#
# # ── Reminder: create the Atlas Vector Search index if not done yet ─────────
# # Index definition (JSON):
# # {
# #   "fields": [
# #     { "type": "vector", "path": "profileEmbedding",
# #       "numDimensions": 768, "similarity": "cosine" },
# #     { "type": "filter", "path": "gender" },
# #     { "type": "filter", "path": "statusTag" },
# #     { "type": "filter", "path": "religion" },
# #     { "type": "filter", "path": "maritalStatus" },
# #     { "type": "filter", "path": "wantKids" },
# #     { "type": "filter", "path": "openToRelocate" },
# #     { "type": "filter", "path": "city" }
# #   ]
# # }

print('Fill in MONGODB_URI above and uncomment the code block to insert directly.')
print('Or use the downloaded JSON with your /api/seed endpoint.')

---
## Quick reference — what changed and why

```python
# ── OLD (text-embedding-004) ─────────────────────────────────────────────────
import google.generativeai as genai
genai.configure(api_key=KEY)
result = genai.embed_content(
    model='models/text-embedding-004',
    content=text,
    task_type='RETRIEVAL_DOCUMENT',   # ← no longer supported on new models
)
vector = result['embedding']          # ← dict access
# dims: 768 (fixed), not normalised for truncated dims

# ── NEW (gemini-embedding-2) ─────────────────────────────────────────────────
from google import genai
from google.genai import types
client = genai.Client(api_key=KEY)
result = client.models.embed_content(
    model='gemini-embedding-2',
    contents=f'title: none | text: {text}',  # ← task instruction in prompt
    config=types.EmbedContentConfig(output_dimensionality=768),
)
vector = list(result.embeddings[0].values)   # ← attribute access, auto-normalised
# dims: 768, auto-normalised (L2 norm ≈ 1.0)
```

### Atlas vector index — must match `numDimensions: 768`
If you previously had an index built for `text-embedding-004` vectors (also 768-dim
but from a **different model space**), you **must delete and recreate** the index
and re-embed all documents. The two models are incompatible — vectors from different
models cannot be meaningfully compared even when they have the same dimension count.
